# App 6 · Anthropic Skills — 当 prompt 长到没法分享的时候

写过两年 LLM 应用的人都会遇到这个问题：你的核心提示词越来越长。一开始 50 字够，三个月后 500 字，半年后 2000 字带 5 个 few-shot 例子和 3 段边界条件说明。同事问你"能不能把这套提示词给我"，你发个 .txt 给他——但他粘进自己的项目还得调，因为你的 prompt 引用了你那边的工具、你的数据格式、你的命名约定。**prompt 不是孤立资产，它和它运行的上下文绑死**。

Anthropic 在 2024 年下半年开始推 [Skills](https://www.anthropic.com/news/agent-skills) 格式就是为了解决这件事。Skill 是一个**文件夹**，不是一个文件——里面有 prompt（`SKILL.md`）、有 helper 脚本（可选）、有按需加载的参考文档（也可选）。整个文件夹可以 git clone、可以放进 `~/.claude/skills/`、可以传到 [Claude.ai marketplace](https://claude.ai/skills)。**它把"prompt + 配套代码 + 文档"打包成一个原子单位**，这才是可分享、可版本化的能力资产。

```
my_skill/
├── SKILL.md           # 必需：YAML frontmatter (name + description) + body
├── helper.py          # 可选：Skill 可调用的脚本
└── reference/         # 可选：按需加载的文档
    └── checklist.md
```

Skills 还有一个杀手特性叫**Progressive Disclosure**——LLM 在不同时机加载不同深度的内容：

1. **启动**：只读所有 Skill 的 `description`（几十字一条），整体几百 token
2. **匹配**：用户来 query，LLM 看 description 决定调哪个 Skill，把那个 Skill 的 `body` 加载进来（几百字）
3. **深入**：执行过程中 Skill body 提到要查某个 reference/*.md，按需再加载

这么做的核心收益是 **省 context = 省钱 + 提速**。如果一个 LLM 客户端连了 50 个 Skills、Skill 又各有 10 页 reference，全量加载是 50 × 10 × 5000 = 250 万 token，每次推理都吃完——progressive disclosure 把它压到典型 query 只用 1-2 个 Skill 各几百 token。

这一节做三件事——理解 Skill 文件夹结构、用 LLM 路由 query 到合适 Skill、手写一个新 Skill 演示完整生命周期。还会看 Skills × MCP 集成模式：Skill 描述 workflow，MCP 提供 tool，两者配合。

> **跑这一节前**：跑过 [App5 MCP](./App5_MCP_Server.ipynb) 理解工具协议。本节 LLM 路由演示需要 `utils.config.setup()` 拿到可用 LLM 后端；离线时也能跑（传 mock LLM）。

In [1]:
# 自动定位 repo 根目录，让 utils 可以 import
import os, sys
_cur = os.path.abspath("")
_root = None
for _c in [_cur, os.path.dirname(_cur), os.path.dirname(os.path.dirname(_cur))]:
    if os.path.isdir(os.path.join(_c, "utils")) and os.path.isfile(os.path.join(_c, "README.md")):
        _root = _c; break
if _root is None:
    raise RuntimeError("找不到 repo 根目录")
os.chdir(_root); sys.path.insert(0, _root)
print(f"📂 repo root: {_root}")


📂 repo root: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code


In [2]:
# 导入：LLM + MCP + Skills helpers
from utils.config import setup
env = setup()
from utils.mcp_helpers import (
    EduMCPServer, EduMCPClient,
    ToolDef, ResourceDef, PromptDef,
    tool_from_function, MCP_AVAILABLE,
)
from utils.skills_helpers import (
    Skill, parse_skill_md, validate_skill,
    discover_skills, match_skill_for_query, load_skill_progressive,
)
import json

llm = env.get_llm()
print(f"✓ LLM 就位")
print(f"✓ MCP SDK 可用: {MCP_AVAILABLE}  (False 走 EduMCPServer 教学模式)")
print(f"✓ Skills helpers 就位")


[OK] 已加载配置: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code\.env
课程环境配置:
  API Key:   ✓ 已配置
  LLM:       dashscope / qwen-plus-2025-01-25
  Embedding: dashscope / text-embedding-v3


[LLM] dashscope / qwen-plus-2025-01-25
✓ LLM 就位
✓ MCP SDK 可用: False  (False 走 EduMCPServer 教学模式)
✓ Skills helpers 就位


In [3]:
# 复用 mcp_server_demo
import sys
from pathlib import Path
sys.path.insert(0, str(Path('Applications/mcp_server_demo')))
from server import build_server as build_demo_server  # type: ignore

demo_server = build_demo_server()
demo_client = EduMCPClient(user_id="alice")
demo_client.connect(demo_server)

print(f"✓ Demo server 就位 ({len(demo_client.list_all_tools())} tools)")
for t in demo_client.list_all_tools():
    print(f"  • {t['name']}: {t['description']}")


✓ Demo server 就位 (3 tools)
  • query_order: Look up an order by ID
  • check_inventory: Check stock quantity for a SKU
  • send_notification: Send a notification to a user


---


## Why Skills + 三协议对比

### 三协议生态

|  | Prompts | MCP | Skills |
|---|---|---|---|
| **解决** | 单次任务说明 | 外部能力接入 | 内化能力打包 |
| **形态** | 字符串 / 模板 | server 暴露 tool | 文件夹 (SKILL.md + scripts) |
| **生命周期** | 一次性 | 长期连接 | 按需载入 |
| **复用范围** | 单 prompt | 多 LLM 共享 | 跨项目跨团队 |
| **关键特性** | 灵活 | 跨厂可移植 | **Progressive Disclosure** |

### 一个 Skill 的样子

```
my_skill/
├── SKILL.md           # 必需：YAML frontmatter + body
├── helper.py          # 可选：Claude 可调用的脚本
└── reference/         # 可选：progressive disclosure 文档
    ├── checklist.md
    └── examples.json
```

`SKILL.md` 头部：
```yaml
---
name: code-review
description: Performs a structured code review... (1-2 句话决定何时被选中)
allowed-tools: [bash, read_file]
version: "0.2"
---

# Code Review Skill
... (body：步骤 / 例子 / 模板)
```

### 关键创新：Progressive Disclosure

**传统 prompt**：把所有指令塞 system prompt → 每次都烧 context
**Skills**：
1. 启动时只读 `description`（几十字）
2. 匹配到再读 `body`（几百字）
3. 需要细节再读 `reference/*.md`（按需）

**省 context = 省钱 + 提速。**


<!-- skill-folder-tour -->
### 先把 Skill 文件夹看清楚

课堂里不用把 Skills 想成抽象概念：它就是一个可复制的文件夹。学员真正需要改的通常只有三处：`SKILL.md` 的 `name`、`description`、以及正文里的 workflow；`helper.py` 和 `reference/` 是进阶扩展。


In [4]:
from pathlib import Path

skills_root = Path("Applications/skills_demo") if Path("Applications/skills_demo").exists() else Path("assets/enterprise_5days/skills_demo")
print(f"Skills 根目录: {skills_root.resolve()}")
print()
print("课堂要看懂的结构：")
for skill_dir in sorted(p for p in skills_root.iterdir() if p.is_dir()):
    files = []
    if (skill_dir / "SKILL.md").exists():
        files.append("SKILL.md")
    files += [p.name for p in skill_dir.glob("*.py")]
    ref_dir = skill_dir / "reference"
    if ref_dir.exists():
        files.append("reference/")
    print(f"  {skill_dir.name}/ -> {', '.join(files)}")

print()
print("最小改造顺序：")
print("  1) 复制一个现成 skill 文件夹")
print("  2) 改 SKILL.md: name + description + workflow")
print("  3) 跑 validate_skill()，再用 match_skill_for_query() 测路由")


Skills 根目录: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code\Applications\skills_demo

课堂要看懂的结构：
  capstone_assistant/ -> SKILL.md, eval.py, pipeline.py, reference/
  code_review/ -> SKILL.md, helper.py, reference/
  db_query/ -> SKILL.md, reference/

最小改造顺序：
  1) 复制一个现成 skill 文件夹
  2) 改 SKILL.md: name + description + workflow
  3) 跑 validate_skill()，再用 match_skill_for_query() 测路由


---

## SKILL.md 解剖 + 写第一个 Skill

`skills_demo/code_review/SKILL.md` 是一个完整例子，看一下：


In [5]:
# 解析现成的 code-review skill
fm, body = parse_skill_md("Applications/skills_demo/code_review/SKILL.md")
print("Frontmatter:")
for k, v in fm.items():
    print(f"  {k}: {v}")
print(f"\nBody (前 400 字):\n{body[:400]}...")

# 也看下 validate
result = validate_skill("Applications/skills_demo/code_review")
print(f"\nvalidate: ok={result['ok']}")
if result['warnings']:
    print(f"  warnings: {result['warnings']}")


Frontmatter:
  name: code-review
  description: 对 Python 代码或 PR 做结构化审查：运行 ruff/mypy，并按 checklist 输出阻塞问题、应改问题和建议。Use when the user asks to review, audit, or check code quality.
  allowed-tools: ['bash', 'read_file']
  version: 0.2

Body (前 400 字):
# Code Review Skill

A pragmatic code reviewer for Python projects. Runs static checks and applies a
team checklist, then produces a prioritized report.

## When to use

- "Review this PR / file / function"
- "Check the code quality of X"
- "Audit this module for issues"

## Workflow

1. **Identify scope**: ask which files / diff to review
2. **Run static checks** (use `helper.py:run_checks()`):
 ...

validate: ok=True


In [6]:
# Discovery: 列出 skills_demo/ 里所有 skill
skills = discover_skills("Applications/skills_demo")
print(f"发现 {len(skills)} 个 skills:")
for s in skills:
    print(f"  • {s.name} (v{s.version}): {s.description[:80]}...")
    print(f"      helpers: {[p.name for p in s.helper_files]}")
    print(f"      references: {[p.name for p in s.reference_files]}")


发现 3 个 skills:
  • enterprise-knowledge-assistant (v1.0): 企业知识助手：回答 HR 政策、产品、技术 API、订单/库存等内部问题，并在 RAG、MCP tools、direct LLM 之间自动路由。Use for ...
      helpers: ['eval.py', 'pipeline.py']
      references: ['architecture.md', 'eval_cases.jsonl']
  • code-review (v0.2): 对 Python 代码或 PR 做结构化审查：运行 ruff/mypy，并按 checklist 输出阻塞问题、应改问题和建议。Use when the use...
      helpers: ['helper.py']
      references: ['checklist.md']
  • db-query (v0.1): 把订单、库存、通知类自然语言问题路由到 enterprise MCP server。Use when the user asks about ORD/SKU/o...
      helpers: []
      references: ['sql_examples.md']


In [7]:
# [LEARNER_FILL] 难度=基础+进阶 | 提示=约30-60行 | 参考实现见 enterprise_5days/student/Day4_下午_MCP与Skills.ipynb
# ============================================================
# 练习 3 | 写一个完整的 SKILL.md
# ============================================================
#
# 【基础】（人人必做，10 min）
#   写一个 meeting_notes skill 的 SKILL.md 字符串：
#   - frontmatter: name + description
#   - body: 何时用 + 步骤 (3-5 步)
#   - validate 通过 (ok=True)
#
# 【进阶】（技术学员选做，10 min）
#   实现 score_skill_description(desc, llm)：
#   - 用 LLM 给 description 打分 1-5（清晰度 + 可路由性）
#   - 返回 (score, suggestion)
#   - 演示：好 description 是技术活
# ============================================================
import tempfile, os


def write_meeting_notes_skill():
    """【基础】返回 SKILL.md 完整字符串"""
    # ↓↓↓ 【基础】填空（约 25 行）↓↓↓
    return """---
name: meeting-notes
description: Helps the user summarize a meeting transcript or audio recording into structured notes — action items, decisions, follow-ups. Use when the user has a meeting recording or transcript and asks for a summary.
allowed-tools: [read_file]
version: "0.1"
---

# Meeting Notes Skill

Turn meeting transcripts into structured, actionable notes.

## When to use

- "总结这场会议"
- "从录音里提取 action items"
- "这场会议有什么决定？"

## Workflow

1. **Identify input**: 文件路径 / 直接粘贴的 transcript
2. **Extract**:
   - **决定 (Decisions)**: 拍板的事项
   - **Action items**: 谁、做什么、何时
   - **Follow-ups**: 待跟进 / 待澄清
3. **Format** as markdown:
   ```markdown
   ## Decisions
   - ...

   ## Action Items
   - [ ] @<owner> <task> by <date>

   ## Follow-ups
   - ...
   ```
4. **Verify** with the user before sending out.

## What this skill does NOT

- Translate / transcribe (use a STT skill)
- Send the notes (use a notification skill)
"""
    # ↑↑↑ 【基础】结束 ↑↑↑


def score_skill_description(desc, llm):
    """【进阶】LLM 评分 description 质量"""
    # ↓↓↓ 【进阶】填空（约 14 行）↓↓↓
    prompt = f"""评分一个 Skill description 的质量 1-5 分。**严格使用全谱**：
- **5 分**：完美 — 清楚触发条件 + 动词宾语 + 30-200 字 + 与其他 skill 明显区分
- **4 分**：良 — 满足 3 项
- **3 分**：及格 — 满足 2 项
- **2 分**：差 — 只满足 1 项（如缺触发条件 / 太短 / 太模糊）
- **1 分**：极差 — 几乎不可用（< 10 字 / 完全模糊 / 无具体动词）

绝对禁止全部 3 分中庸。差的就给 1-2，好的就给 4-5。

description: {desc}

输出格式（严格 2 行）：
SCORE: <数字>
SUGGESTION: <一句话改进建议>"""
    raw = llm.generate(prompt, temperature=0).strip()
    score = 3
    suggestion = ""
    for line in raw.splitlines():
        if line.upper().startswith("SCORE:"):
            try:
                score = int("".join(c for c in line if c.isdigit())[:1])
            except (ValueError, IndexError):
                pass
        elif line.upper().startswith("SUGGESTION:"):
            suggestion = line.split(":", 1)[1].strip()
    return (score, suggestion)
    # ↑↑↑ 【进阶】结束 ↑↑↑


def verify():
    print("=" * 56); print("【基础】write_meeting_notes_skill"); print("=" * 56)
    try:
        skill_md = write_meeting_notes_skill()
        # 写到临时文件夹验证
        with tempfile.TemporaryDirectory() as tmp:
            sd = os.path.join(tmp, "meeting-notes")
            os.makedirs(sd)
            with open(os.path.join(sd, "SKILL.md"), "w", encoding="utf-8") as f:
                f.write(skill_md)
            result = validate_skill(sd)
            print(f"  validate: ok={result['ok']}, warnings={result['warnings']}")
            assert result["ok"], f"validate 不通过：{result['errors']}"
            fm, body = parse_skill_md(os.path.join(sd, "SKILL.md"))
            assert fm["name"] == "meeting-notes"
            assert len(fm["description"]) > 30
            print(f"  name: {fm['name']}  desc 长度: {len(fm['description'])}  body 长度: {len(body)}")
        print("✅ 基础通过\n")
    except NotImplementedError:
        print("⏭ 基础未实现\n"); return
    except Exception as e:
        print(f"❌ 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】score_skill_description"); print("=" * 56)
    try:
        good = "Helps the user write structured meeting notes from transcripts. Extracts action items, decisions, and follow-ups. Use when the user has a meeting recording or transcript."
        bad = "AI 助手"
        score_g, sug_g = score_skill_description(good, llm)
        score_b, sug_b = score_skill_description(bad, llm)
        print(f"  好 desc: score={score_g} | suggestion: {sug_g[:80]}")
        print(f"  差 desc: score={score_b} | suggestion: {sug_b[:80]}")
        assert score_g >= score_b
        print("✅ 进阶通过 — 好 description 评分更高")
    except NotImplementedError:
        print("⏭ 进阶跳过")
    except Exception as e:
        print(f"❌ 进阶未通过: {type(e).__name__}: {e}")

verify()


【基础】write_meeting_notes_skill
  validate: ok=True, warnings=[]
  name: meeting-notes  desc 长度: 205  body 长度: 641
✅ 基础通过

【进阶】score_skill_description


  好 desc: score=5 | suggestion: 描述已很清晰，无需改进。
  差 desc: score=1 | suggestion: 添加具体触发条件、清晰任务描述及动词宾语以提高明确性。
✅ 进阶通过 — 好 description 评分更高


---

## Helper Scripts + Progressive Disclosure

### Helper Scripts

Skills 不是死文档——可以挂 Python 脚本。Claude 会读 `helper.py` 的 docstring，按需调用其中函数。

例：`code_review/helper.py` 提供 `run_checks(target)` 跑 ruff + mypy。Skill body 教 Claude 在第 2 步调用它。

### Progressive Disclosure（杀手特性）

**Naive 做法**：所有 skill 全部内容塞进 system prompt → context 爆掉

**Skills 做法**：3 层载入
1. **Always**: discovery 时只读 `description`（几十字）
2. **On match**: 用户 query 匹配 → 加载 body（几百字）
3. **On demand**: query 涉及细节 → 加载相关 `reference/*.md`（按需）

下面演示 `load_skill_progressive`：


In [8]:
# Progressive disclosure 实战
skills = discover_skills("Applications/skills_demo")
code_review = next(s for s in skills if s.name == "code-review")

# 场景 A：query 简单 → 只载入 body，不读 reference/checklist.md
loaded_a = load_skill_progressive(code_review, "review this function: def add(a,b): return a+b", llm)
print("=" * 56)
print("场景 A: '简单函数 review'")
print("=" * 56)
print(f"  loaded references: {[r['name'] for r in loaded_a['references_loaded']]}")
print(f"  estimated tokens: {loaded_a['tokens_estimate']}")

# 场景 B：query 复杂 + 提到团队 → LLM 决定加载 checklist.md
loaded_b = load_skill_progressive(
    code_review,
    "review this PR — 我们团队对错误处理特别敏感，要走完整 checklist",
    llm,
)
print("\n" + "=" * 56)
print("场景 B: '完整 checklist review'")
print("=" * 56)
print(f"  loaded references: {[r['name'] for r in loaded_b['references_loaded']]}")
print(f"  estimated tokens: {loaded_b['tokens_estimate']}")
print(f"\n💡 场景 A 省了 ~{loaded_b['tokens_estimate'] - loaded_a['tokens_estimate']} tokens — 简单任务不需要全文档")


场景 A: '简单函数 review'
  loaded references: ['checklist.md']
  estimated tokens: 540



场景 B: '完整 checklist review'
  loaded references: ['checklist.md']
  estimated tokens: 540

💡 场景 A 省了 ~0 tokens — 简单任务不需要全文档


In [9]:
# [LEARNER_FILL] 难度=基础+进阶 | 提示=约20-50行 | 参考实现见 enterprise_5days/student/Day4_下午_MCP与Skills.ipynb
# ============================================================
# 练习 4 | validate_skill + LLM 路由 (match_skill_for_query)
# ============================================================
#
# 【基础】（人人必做，10 min）
#   实现 validate_all(skills_dir)：跑 validate_skill 检查所有子目录，返回 dict {name: {ok, errors, warnings}}
#
# 【进阶】（技术学员选做，15 min）
#   实现 audit_routing(test_queries, skills, llm)：
#   - 一组 (query, expected_skill_name) pairs
#   - 用 match_skill_for_query 路由，统计准确率
#   - 错路由 case 列出来供 description 改进
# ============================================================

def validate_all(skills_dir):
    """【基础】批量 validate"""
    # ↓↓↓ 【基础】填空（约 6 行）↓↓↓
    from pathlib import Path as P
    out = {}
    for sub in sorted(P(skills_dir).iterdir()):
        if sub.is_dir() and (sub / "SKILL.md").exists():
            out[sub.name] = validate_skill(sub)
    return out
    # ↑↑↑ 【基础】结束 ↑↑↑


def audit_routing(test_queries, skills, llm):
    """【进阶】路由准确率审计"""
    # ↓↓↓ 【进阶】填空（约 14 行）↓↓↓
    correct = 0
    audit_log = []
    for query, expected in test_queries:
        picked = match_skill_for_query(query, skills, llm)
        picked_name = picked.name if picked else None
        ok = picked_name == expected
        if ok:
            correct += 1
        audit_log.append({
            "query": query,
            "expected": expected,
            "picked": picked_name,
            "ok": ok,
        })
    return {
        "accuracy": correct / len(test_queries) if test_queries else 0,
        "errors": [a for a in audit_log if not a["ok"]],
        "log": audit_log,
    }
    # ↑↑↑ 【进阶】结束 ↑↑↑


def verify():
    print("=" * 56); print("【基础】validate_all('Applications/skills_demo')"); print("=" * 56)
    try:
        report = validate_all("Applications/skills_demo")
        for name, r in report.items():
            status = "✓" if r["ok"] else "✗"
            print(f"  {status} {name}: errors={r['errors']}, warnings={len(r['warnings'])}")
        assert all(r["ok"] for r in report.values()), "应所有 demo skill 都 valid"
        print("✅ 基础通过\n")
    except NotImplementedError:
        print("⏭ 基础未实现\n"); return
    except Exception as e:
        print(f"❌ 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】audit_routing"); print("=" * 56)
    try:
        skills = discover_skills("Applications/skills_demo")
        test_cases = [
            ("review my python code", "code-review"),
            ("查 ORD-005 订单", "db-query"),
            ("入职 5 年年假几天", "enterprise-knowledge-assistant"),
            ("SKU-A100 库存多少", "db-query"),
        ]
        result = audit_routing(test_cases, skills, llm)
        print(f"  路由准确率: {result['accuracy']:.0%}")
        for log in result["log"]:
            mark = "✓" if log["ok"] else "✗"
            print(f"    {mark} '{log['query'][:40]}' → expected={log['expected']}, picked={log['picked']}")
        if result["errors"]:
            print(f"\n  💡 错路由 case 是改 description 的输入")
        print("✅ 进阶通过")
    except NotImplementedError:
        print("⏭ 进阶跳过")
    except Exception as e:
        print(f"❌ 进阶未通过: {type(e).__name__}: {e}")

verify()


【基础】validate_all('Applications/skills_demo')
  ✓ capstone_assistant: errors=[], warnings=0
  ✓ code_review: errors=[], warnings=0
  ✓ db_query: errors=[], warnings=0
✅ 基础通过

【进阶】audit_routing


  路由准确率: 100%
    ✓ 'review my python code' → expected=code-review, picked=code-review
    ✓ '查 ORD-005 订单' → expected=db-query, picked=db-query
    ✓ '入职 5 年年假几天' → expected=enterprise-knowledge-assistant, picked=enterprise-knowledge-assistant
    ✓ 'SKU-A100 库存多少' → expected=db-query, picked=db-query
✅ 进阶通过


---

## Skills × MCP 集成模式

```
┌─────────────────┐         ┌──────────────────┐
│  Skills          │         │  MCP             │
│  (能力 / 何时做) │ ─────▶ │  (工具 / 怎么调) │
└─────────────────┘         └──────────────────┘

例：db-query Skill 教 Claude『查订单』，actual API 调用走 enterprise-demo MCP server
```

`skills_demo/db_query/SKILL.md` 的 `allowed-tools` 字段限制了它只能调 3 个 MCP tool：
- `mcp__enterprise-demo__query_order`
- `mcp__enterprise-demo__check_inventory`
- `mcp__enterprise-demo__send_notification`

下面看完整流程。


In [10]:
# 完整流程：用户 query → 路由到 Skill → Skill 调 MCP → 返回
skills = discover_skills("Applications/skills_demo")
demo_server = build_demo_server()  # 复用前面的 MCP server
demo_client = EduMCPClient(user_id="demo")
demo_client.connect(demo_server)


def skill_calls_mcp(query):
    """端到端：query → 选 skill → 让 Claude 按 skill body 决定调 MCP tool"""
    # 1. Skill discovery + routing
    skill = match_skill_for_query(query, skills, llm)
    if skill is None:
        return f"[no matching skill] {query}"
    # 2. Load skill body (progressive)
    loaded = load_skill_progressive(skill, query, llm)
    # 3. Skill body 指导 Claude 调哪个 MCP tool
    tools = demo_client.list_all_tools()
    desc = "\n".join(f"- {t['name']}({list(t['parameters']['properties'].keys())})" for t in tools)
    plan_prompt = f'''Skill: {skill.name}
Skill 指导:
{loaded['body'][:600]}

可用 MCP tools:
{desc}

用户 query: {query}

按 skill 指导决定调哪个 tool。输出 JSON: {{"tool": "...", "arguments": {{...}}}}。'''
    raw = llm.generate(plan_prompt, temperature=0).strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    try:
        plan = json.loads(raw)
        result = demo_client.call(demo_server.name, plan["tool"], **plan["arguments"])
        return {"skill": skill.name, "tool": plan["tool"], "result": result}
    except Exception as e:
        return {"skill": skill.name, "error": str(e)}


# Demo
print("=" * 60); print("Demo 1: '查 ORD-002'"); print("=" * 60)
print(json.dumps(skill_calls_mcp("查 ORD-002 订单状态"), ensure_ascii=False, indent=2))

print("\n" + "=" * 60); print("Demo 2: '库存查 SKU-A100'"); print("=" * 60)
print(json.dumps(skill_calls_mcp("SKU-A100 还有多少库存？"), ensure_ascii=False, indent=2))


Demo 1: '查 ORD-002'


{
  "skill": "db-query",
  "error": "Expecting value: line 1 column 1 (char 0)"
}

Demo 2: '库存查 SKU-A100'


{
  "skill": "db-query",
  "error": "Expecting value: line 1 column 1 (char 0)"
}


In [11]:
# [LEARNER_FILL] 难度=基础+进阶 | 提示=约30-60行 | 参考实现见 enterprise_5days/student/Day4_下午_MCP与Skills.ipynb
# ============================================================
# 练习 5 | 写一个新 Skill 调用现有 MCP server
# ============================================================
#
# 【基础】（人人必做，10 min）
#   写 notify_skill_md：返回字符串 — 一个 notify-customer skill
#   - description 提到『何时用：发通知 / 提醒 / 告知』
#   - body 教 Claude 用 send_notification tool
#   - allowed-tools 含 mcp__enterprise-demo__send_notification
#
# 【进阶】（技术学员选做，15 min）
#   写 multi_tool_skill_md：一个 skill 调 ≥ 2 个 MCP tool
#   场景：『发货前自动检查 + 通知』
#   - 先 check_inventory(sku)
#   - 库存够 → query_order(order_id) 拿客户名
#   - 然后 send_notification 给客户
#   body 要清楚教 Claude 这个 3 步流程
# ============================================================

def notify_skill_md():
    """【基础】返回 SKILL.md 字符串"""
    # ↓↓↓ 【基础】填空（约 16 行）↓↓↓
    return """---
name: notify-customer
description: Sends a notification to a customer/user via the enterprise notification system. Use when the user wants to inform, alert, or remind a specific person about something (order shipped, inventory ready, account issue, etc).
allowed-tools: [mcp__enterprise-demo__send_notification]
version: "0.1"
---

# Notify Customer Skill

Send a single notification to a user.

## When to use

- "通知 alice 她的订单到了"
- "发个提醒给 bob"
- "告诉 carol 库存已补"

## Workflow

1. 解析: 谁 (user_id) + 内容 (message)
2. 调 `mcp__enterprise-demo__send_notification(user_id=..., message=...)`
3. 确认发出 + 反馈给用户

## Examples

| 用户说 | tool 调用 |
|---|---|
| "通知 alice 订单到了" | `send_notification(user_id="alice", message="订单到了")` |
| "提醒 bob 续费" | `send_notification(user_id="bob", message="请续费")` |
"""
    # ↑↑↑ 【基础】结束 ↑↑↑


def multi_tool_skill_md():
    """【进阶】3 步发货 skill"""
    # ↓↓↓ 【进阶】填空（约 24 行）↓↓↓
    return """---
name: pre-ship-check
description: Performs the pre-shipment workflow — checks inventory, looks up the customer for an order, then notifies them that the order is ready to ship. Use when the user wants to do a "ready to ship" or "pre-ship" workflow for an existing order.
allowed-tools:
  - mcp__enterprise-demo__check_inventory
  - mcp__enterprise-demo__query_order
  - mcp__enterprise-demo__send_notification
version: "0.1"
---

# Pre-Ship Check Skill

3-step workflow: 检库存 → 查客户 → 通知。

## When to use

- "ORD-001 准备发货"
- "检查 ORD-XXX 是否能发货并通知客户"
- "走发货前检查"

## Workflow (强制 3 步顺序)

1. **Check inventory**: 调 `check_inventory(sku=...)`
   - 库存 = 0 → 中止，告诉用户『缺货，无法发货』
2. **Look up order**: 调 `query_order(order_id=...)` 拿 customer 字段
3. **Notify**: 调 `send_notification(user_id=<customer>, message="您的订单 <id> 即将发货")`

## Error handling

- 任何一步失败 → 不继续后面步骤，回滚之前的 side effects（这里只 send_notification 有 side effect）
- 报告每步 PASS / FAIL 状态

## Output format

```
[1/3] check_inventory(SKU-XXX) → in_stock: <yes/no>
[2/3] query_order(ORD-XXX) → customer: <name>
[3/3] send_notification(<name>, ...) → sent
```
"""
    # ↑↑↑ 【进阶】结束 ↑↑↑


def verify():
    print("=" * 56); print("【基础】notify_skill_md"); print("=" * 56)
    try:
        md = notify_skill_md()
        with tempfile.TemporaryDirectory() as tmp:
            sd = os.path.join(tmp, "notify-customer")
            os.makedirs(sd)
            with open(os.path.join(sd, "SKILL.md"), "w", encoding="utf-8") as f:
                f.write(md)
            result = validate_skill(sd)
            assert result["ok"], result["errors"]
            fm, body = parse_skill_md(os.path.join(sd, "SKILL.md"))
            assert "send_notification" in fm.get("allowed-tools", [])[0]
            assert "通知" in body or "notify" in body.lower()
        print(f"  validate ok + 含 send_notification + body 含通知关键词")
        print("✅ 基础通过\n")
    except NotImplementedError:
        print("⏭ 基础未实现\n"); return
    except Exception as e:
        print(f"❌ 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】multi_tool_skill_md"); print("=" * 56)
    try:
        md = multi_tool_skill_md()
        # 检查 body 含三步流程关键词
        for kw in ["check_inventory", "query_order", "send_notification"]:
            assert kw in md, f"应包含 {kw}"
        # frontmatter 含 3 个 allowed-tools (multiline 形式)
        assert md.count("mcp__enterprise-demo__") >= 3, "应有 3 个 MCP tool"
        print("  ✓ body 含三步流程")
        print("  ✓ allowed-tools 含 3 个 MCP tools")
        print("✅ 进阶通过 — 复合 skill 适合多步业务流程")
    except NotImplementedError:
        print("⏭ 进阶跳过")
    except Exception as e:
        print(f"❌ 进阶未通过: {type(e).__name__}: {e}")

verify()


【基础】notify_skill_md
  validate ok + 含 send_notification + body 含通知关键词
✅ 基础通过

【进阶】multi_tool_skill_md
  ✓ body 含三步流程
  ✓ allowed-tools 含 3 个 MCP tools
✅ 进阶通过 — 复合 skill 适合多步业务流程


---

## 实操：用 code_review Skill 审查一段含 bug 的代码

到这里学员可能仍觉得 Skills "概念懂了，但具体能给我做什么？"

下面用 `code_review/` skill 跑一个**真实任务**：

```
任务情景: 我有一段 Python 代码，疑似有 bug + 风格问题。
        想要 Claude 按团队 checklist 帮我审一遍。

无 skill 怎么办: 写个 prompt"review this code" → 输出格式不稳，
                每个 reviewer 可能给不同维度的反馈。

有 skill 怎么办: discover code-review skill → 自动按 SKILL.md
                的 5 步 workflow + checklist.md 出结构化报告。
```


In [12]:
# 实操 1: 用 code_review skill 端到端审查
import sys, json
from pathlib import Path

# 待审的代码（含若干典型问题）
buggy_code = """
def calc_discount(price, discount):
    # 没参数校验、负价格不报错、整数除法可能出问题
    final = price - price * discount
    return final

def process_orders(orders):
    # 异常被吞、空列表会报错
    total = 0
    try:
        for o in orders:
            total = total + o['amount']
    except:
        pass
    return total / len(orders)

def get_user(user_id):
    # SQL 注入、明文密码
    sql = "SELECT * FROM users WHERE id = " + str(user_id)
    return execute(sql)
"""

# Step 1: discover + 路由到 code-review
skills = discover_skills("Applications/skills_demo")
picked = match_skill_for_query("review this Python code for bugs and style issues", skills, llm)
print(f"Step 1 路由: {picked.name if picked else '(none)'}")

# Step 2: progressive load — 简单 query 不需 reference
loaded = load_skill_progressive(picked, "review this Python code", llm)
print(f"Step 2 加载: body {len(loaded['body'])} 字符 + {len(loaded['references_loaded'])} 个 reference")

# Step 3: 让 LLM 按 skill body 跑审查 workflow
review_prompt = f"""你是 code-review skill 的执行体。严格按下面的 workflow 审查代码。

[SKILL body]
{loaded['body']}

[要审的代码]
```python
{buggy_code}
```

按 workflow 第 4 步的 Output format 输出结构化报告。
"""
report = llm.generate(review_prompt, temperature=0.1)
print("\n" + "=" * 60)
print("Step 3 输出（按 SKILL workflow 的结构化报告）:")
print("=" * 60)
print(report)


Step 1 路由: code-review


Step 2 加载: body 1268 字符 + 1 个 reference



Step 3 输出（按 SKILL workflow 的结构化报告）:
```markdown
## Code Review Summary

**Files reviewed**: [calc_discount, process_orders, get_user]  
**Critical**: 3  **Important**: 2  **Nits**: 0

### 🔴 Blocking
- `get_user`: SQL injection vulnerability due to string concatenation in query construction. This is a severe security risk.
- `process_orders`: Division by zero error when `orders` is an empty list. This will raise a `ZeroDivisionError`.
- `process_orders`: Swallowing all exceptions without logging or handling specific errors can hide critical issues.

### 🟡 Should fix
- `calc_discount`: Missing input validation for negative prices or invalid discount values. This could lead to incorrect calculations.
- `calc_discount`: Potential precision loss due to integer division if inputs are integers. Consider using floating-point arithmetic explicitly.

### 🟢 Nice to have
- None identified in this review.
```

### Explanation of Findings:
1. **SQL Injection (`get_user`)**: Concatenating user input

In [13]:
# 实操 2: 对比 — naive 跑 3 次看格式漂移 vs skill 跑 3 次看格式稳定
# 关键洞察：单次跑 naive 也能给好答案；但**多次跑**naive 输出格式漂移大，
# 不能给下游系统消费。Skill 强制走 SKILL.md workflow 输出可重复。

def has_critical_section(text):
    """检查是否含『阻塞级问题』标识（任何形式）"""
    return any(m in text for m in ["🔴", "Blocking", "Critical", "严重", "BLOCKER"])

def has_should_fix(text):
    return any(m in text for m in ["🟡", "Should fix", "应改", "SHOULD"])

def has_nice(text):
    return any(m in text for m in ["🟢", "Nice", "建议", "NIT"])

def count_bullet_sections(text):
    """大致数 markdown 主标题数量"""
    return sum(1 for line in text.split("\n") if line.strip().startswith(("##", "###")))


print("=" * 70)
print("跑 3 次同代码 — 对比 naive 与 skill-driven 的格式稳定性")
print("=" * 70)

naive_results = []
skill_results = []
for i in range(3):
    print(f"\n第 {i+1} 次...")
    # naive
    n = llm.generate(f"Review this code:\n```python\n{buggy_code}\n```", temperature=0.3)
    naive_results.append(n)
    # skill (复用上面 Step 3 的 review_prompt)
    s = llm.generate(review_prompt, temperature=0.3)
    skill_results.append(s)

# 量化指标
print("\n" + "=" * 70)
print(f"{'指标':<30} {'naive 3 次':<15} {'skill 3 次':<15}")
print("=" * 70)
metrics = [
    ("含『🔴 Blocking』标识", has_critical_section),
    ("含『🟡 Should fix』标识", has_should_fix),
    ("含『🟢 Nice』标识", has_nice),
]
for label, fn in metrics:
    n_n = sum(fn(r) for r in naive_results)
    n_s = sum(fn(r) for r in skill_results)
    print(f"  {label:<28} {n_n:>3}/3 次          {n_s:>3}/3 次")

# 长度方差（格式稳定性的代理指标）
import statistics
naive_lens = [len(r) for r in naive_results]
skill_lens = [len(r) for r in skill_results]
print(f"  长度均值                       {statistics.mean(naive_lens):>6.0f}        {statistics.mean(skill_lens):>6.0f}")
print(f"  长度标准差（越小越稳定）       {statistics.stdev(naive_lens):>6.0f}        {statistics.stdev(skill_lens):>6.0f}")
print(f"  章节数（## 标题）均值          {statistics.mean([count_bullet_sections(r) for r in naive_results]):>6.1f}        {statistics.mean([count_bullet_sections(r) for r in skill_results]):>6.1f}")

print("""
💡 关键观察:
  - 单次看 naive 也写得不错——但同样问题跑 3 次，格式漂移大（章节数 / 标题命名 / 三档分类不稳）
  - skill-driven 强制走 SKILL.md 的 workflow 第 4 步 Output format → 三档分类 + 章节稳定
  - 工业场景下下游系统（dashboard / 自动化流转）需要稳定的结构化输出 → skill 才靠谱
"""[1:])


跑 3 次同代码 — 对比 naive 与 skill-driven 的格式稳定性

第 1 次...



第 2 次...



第 3 次...



指标                             naive 3 次       skill 3 次      
  含『🔴 Blocking』标识                0/3 次            3/3 次
  含『🟡 Should fix』标识              0/3 次            3/3 次
  含『🟢 Nice』标识                    0/3 次            3/3 次
  长度均值                         5090          1534
  长度标准差（越小越稳定）          773            91
  章节数（## 标题）均值            11.3           5.0
💡 关键观察:
  - 单次看 naive 也写得不错——但同样问题跑 3 次，格式漂移大（章节数 / 标题命名 / 三档分类不稳）
  - skill-driven 强制走 SKILL.md 的 workflow 第 4 步 Output format → 三档分类 + 章节稳定
  - 工业场景下下游系统（dashboard / 自动化流转）需要稳定的结构化输出 → skill 才靠谱



### 何时 Skill 真正值得？

| 场景 | 推荐 |
|---|---|
| 一次性 / ad-hoc 任务 | 直接 prompt，不用 skill |
| **团队反复跑同一类工作流**（code review / 周报 / 会议纪要 / SOP）| ✅ Skill |
| 需要**结构化输出**给下游系统消费 | ✅ Skill |
| 需要**可观测 / 可审计**（每次走相同步骤）| ✅ Skill |
| 任务需要**多个 helper script 协作** | ✅ Skill |
| Prompt > 200 字 + 含步骤 + 含示例 | ✅ Skill（已经够复杂了） |

**rule of thumb**: 同样的指令你写给同事第 3 遍 → 该封 Skill 了。

### 用 Skill 你刚才完成了什么

你刚刚把一个**团队 Code Review SOP** 用 1 个 SKILL.md + 1 个 helper.py + 1 个 checklist.md 表达出来。任何团队成员（或他们的 Claude / Cursor / Claude Code）只要 `import` 这个 skill 文件夹就能复用——**不需要重写 prompt，不需要培训新人**。


## 5. 总结

- **Skills = 可打包内化能力**：跨项目跨团队复用
- **三层载入** (progressive disclosure)：name+description always；body 匹配后；reference 按需 → 省 context
- **配 MCP**：Skill 描述 workflow，MCP 提供 tool，两者解耦
- **生产**: 团队共享 git repo / 个人 ~/.claude/skills / Claude.ai marketplace

**下一步**:
- App7_LLMOps — observability + trace + cost
- 进阶练习：写 meeting-notes / score description / Skills × MCP 集成（在 capstone_assistant 范例里有完整实现）
